# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damiannogueira/flyrank-ml-intership/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Colab setup

This setup securely loads the Hugging Face access token from Colab Secrets and prepares DuckDB to query the gated FlyRank warehouse without exposing the token in the notebook.

In [1]:
# Load the Hugging Face token securely from Colab Secrets.
# The token is never printed or stored directly in the notebook.

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [2]:
# Install DuckDB and open a connection to the remote warehouse.
# The Hugging Face token is registered securely without printing it.

%pip -q install duckdb

import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connection ready.")

DuckDB connection ready.


In [3]:
# Define the March 2026 warehouse partition used for the signal checks.

MARCH_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [4]:
# Build one page-level analysis frame for both signal checks.
# March 1-21 contains only pre-decision signals.
# March 22-31 is used only to evaluate the later outcome.

signal_frame = con.sql(f"""
    WITH pre_decision AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_21d,
            SUM(gsc_clicks) AS clicks_21d,
            SUM(gsc_sum_position)
                / NULLIF(SUM(gsc_impressions), 0) AS avg_position_21d
        FROM {MARCH_REL}
        WHERE
            gsc_data_available IS TRUE
            AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
        GROUP BY
            client_hash_id,
            content_hash_id
        HAVING SUM(gsc_impressions) > 0
    ),

    later_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future_impressions_10d
        FROM {MARCH_REL}
        WHERE
            gsc_data_available IS TRUE
            AND report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.impressions_21d,
        p.clicks_21d,
        p.avg_position_21d,
        f.future_impressions_10d
    FROM pre_decision AS p
    INNER JOIN later_window AS f
        USING (client_hash_id, content_hash_id)
""").df()

# Create the future outcome used only to evaluate the signals.
signal_frame["future_change_pct"] = (
    (
        signal_frame["future_impressions_10d"] / 10
        - signal_frame["impressions_21d"] / 21
    )
    / (signal_frame["impressions_21d"] / 21)
    * 100
)

signal_frame["future_decline_target"] = (
    signal_frame["future_change_pct"] < -20
).astype(int)

print(f"Pages available for signal checks: {len(signal_frame):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages available for signal checks: 145,091


In [5]:
# Import pandas to create readable signal buckets.
import pandas as pd

# Signal check 1: bucket pages by pre-decision search volume.
# Print n and the observed future decline rate for each bucket.

signal_frame["volume_bucket"] = pd.cut(
    signal_frame["impressions_21d"],
    bins=[0, 100, 500, 2500, float("inf")],
    labels=["<100", "100-499", "500-2499", "2500+"],
    right=False
)

volume_check = (
    signal_frame
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        future_decline_rate=("future_decline_target", "mean")
    )
    .reset_index()
)

volume_check["future_decline_rate"] = (
    volume_check["future_decline_rate"] * 100
).round(1)

volume_check

,volume_bucket,n,future_decline_rate
0,<100,56633,28.9
1,100-499,38073,33.7
2,500-2499,33362,29.2
3,2500+,17023,25.6


### Signal check 1 — Search volume

This signal is linked to FlyRank's quick-win logic: pages with more search visibility have more potential impact if they are worth reviewing.

I will bucket pages by pre-decision search volume and compare the observed future decline rate across buckets.

**Verdict: MIXED**

Future decline does not increase consistently with search volume. The `100–499` bucket has the highest observed decline rate (33.7%), while the highest-volume `2500+` bucket has the lowest (25.6%).

This does not support treating volume alone as a decline-risk signal. I will keep search volume as an impact/prioritization signal rather than interpret higher impressions as evidence that a page is more likely to decline.

### Signal check 2 — Search position

Search position is a pre-decision visibility signal. I will bucket pages by their weighted average GSC position and compare the observed future decline rate across those buckets.

In [6]:
# Signal check 2: bucket pages by pre-decision average search position.
# Print n and the observed future decline rate for each bucket.

signal_frame["position_bucket"] = pd.cut(
    signal_frame["avg_position_21d"],
    bins=[0, 4, 10, 20, float("inf")],
    labels=["1-3.9", "4-9.9", "10-19.9", "20+"],
    right=False
)

position_check = (
    signal_frame
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        future_decline_rate=("future_decline_target", "mean")
    )
    .reset_index()
)

position_check["future_decline_rate"] = (
    position_check["future_decline_rate"] * 100
).round(1)

position_check

,position_bucket,n,future_decline_rate
0,1-3.9,25236,27.9
1,4-9.9,55540,31.4
2,10-19.9,25950,29.3
3,20+,38365,29.4


**Verdict: MIXED**

Future decline does not increase consistently as average search position worsens. The `4–9.9` bucket has the highest observed decline rate (31.4%), while the lower-ranking buckets are slightly lower.

This does not support using average position alone as a direct decline-risk signal. I will treat position as context for prioritization rather than as evidence that a page is necessarily more likely to decline.

### Baseline rule

The two signal checks were both MIXED, so I will not treat search volume or average position as direct evidence of future decline.

Instead, I will use them as transparent opportunity signals.

A page is worth prioritizing for review when:

- it had at least 500 GSC impressions during the pre-decision window, meaning it already had meaningful search visibility; and
- its weighted average search position was at least 4 but below 20, meaning it was visible in search but was not already consistently near the very top.

Among pages that meet both conditions, higher search volume receives a higher baseline score.

**Reason code:** `visible_mid_rank`

**Action label:** `REVIEW_REFRESH`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Build the transparent baseline queue using only pre-decision signals.
# Future-window columns and labels are deliberately excluded.

baseline_queue = signal_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_21d",
        "clicks_21d",
        "avg_position_21d",
    ]
].copy()

# Apply the single baseline rule.
baseline_queue["qualifies"] = (
    (baseline_queue["impressions_21d"] >= 500)
    & (baseline_queue["avg_position_21d"] >= 4)
    & (baseline_queue["avg_position_21d"] < 20)
)

# Score qualifying pages by their observed pre-decision search visibility.
baseline_queue["baseline_score"] = (
    baseline_queue["qualifies"].astype(int)
    * baseline_queue["impressions_21d"]
)

# Keep one reason code and one action label for selected pages.
baseline_queue["reason_code"] = baseline_queue["qualifies"].map(
    {True: "visible_mid_rank", False: ""}
)

baseline_queue["action_label"] = baseline_queue["qualifies"].map(
    {True: "REVIEW_REFRESH", False: ""}
)

# Keep only pages selected by the rule and rank highest score first.
ranked_queue = (
    baseline_queue[baseline_queue["qualifies"]]
    .drop(columns="qualifies")
    .sort_values(
        ["baseline_score", "client_hash_id", "content_hash_id"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

ranked_queue.insert(0, "rank", range(1, len(ranked_queue) + 1))

print(f"Pages selected by baseline rule: {len(ranked_queue):,}")

ranked_queue.head(10)


Pages selected by baseline rule: 28,430


,rank,client_hash_id,content_hash_id,impressions_21d,clicks_21d,avg_position_21d,baseline_score,reason_code,action_label
0,1,client_23a62021009f63c4,content_e8a52cf3d5988c07,179025.0,472.0,15.593699,179025.0,visible_mid_rank,REVIEW_REFRESH
1,2,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,127816.0,1532.0,4.322690,127816.0,visible_mid_rank,REVIEW_REFRESH
2,3,client_62f4a7e64f5e0096,content_b99ea6861864dea5,125082.0,245.0,4.448818,125082.0,visible_mid_rank,REVIEW_REFRESH
3,4,client_73cda7b4e4f265ea,content_471d9cabce329a66,110663.0,275.0,4.544301,110663.0,visible_mid_rank,REVIEW_REFRESH
4,5,client_62f4a7e64f5e0096,content_7c6373141eae744a,100270.0,57.0,5.813135,100270.0,visible_mid_rank,REVIEW_REFRESH
5,6,client_20259bd6705d81d4,content_82e35c4845e6c391,96942.0,38.0,18.343680,96942.0,visible_mid_rank,REVIEW_REFRESH
6,7,client_23a62021009f63c4,content_5e1c049f62e33b11,89534.0,135.0,17.834231,89534.0,visible_mid_rank,REVIEW_REFRESH
7,8,client_62f4a7e64f5e0096,content_f6116743b00afc2d,81123.0,10.0,9.857661,81123.0,visible_mid_rank,REVIEW_REFRESH
8,9,client_73cda7b4e4f265ea,content_f43118e089ecc69a,80299.0,129.0,4.853248,80299.0,visible_mid_rank,REVIEW_REFRESH
9,10,client_73cda7b4e4f265ea,content_95ff62babbfac9c7,78330.0,155.0,4.581667,78330.0,visible_mid_rank,REVIEW_REFRESH


In [8]:
# Write the ranked baseline queue to the required output path.
# The CSV is regenerated by the notebook and is intentionally not committed to git.

from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

ranked_queue.to_csv(output_path, index=False)

print(f"CSV written to: {output_path}")
print(f"Rows written: {len(ranked_queue):,}")

CSV written to: work/outputs/baseline_action_score.csv
Rows written: 28,430


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# Inspect the top 10 with a simple click-through-rate context.
# CTR here means clicks divided by impressions during the pre-decision window.

top10_review = ranked_queue.head(10).copy()

top10_review["ctr_21d_pct"] = (
    100.0
    * top10_review["clicks_21d"]
    / top10_review["impressions_21d"]
).round(3)

top10_review[
    [
        "rank",
        "content_hash_id",
        "impressions_21d",
        "clicks_21d",
        "ctr_21d_pct",
        "avg_position_21d",
        "reason_code",
        "action_label",
    ]
]


,rank,content_hash_id,impressions_21d,clicks_21d,ctr_21d_pct,avg_position_21d,reason_code,action_label
0,1,content_e8a52cf3d5988c07,179025.0,472.0,0.264,15.593699,visible_mid_rank,REVIEW_REFRESH
1,2,content_e7b5dd4dff461ad2,127816.0,1532.0,1.199,4.322690,visible_mid_rank,REVIEW_REFRESH
2,3,content_b99ea6861864dea5,125082.0,245.0,0.196,4.448818,visible_mid_rank,REVIEW_REFRESH
3,4,content_471d9cabce329a66,110663.0,275.0,0.249,4.544301,visible_mid_rank,REVIEW_REFRESH
4,5,content_7c6373141eae744a,100270.0,57.0,0.057,5.813135,visible_mid_rank,REVIEW_REFRESH
5,6,content_82e35c4845e6c391,96942.0,38.0,0.039,18.343680,visible_mid_rank,REVIEW_REFRESH
6,7,content_5e1c049f62e33b11,89534.0,135.0,0.151,17.834231,visible_mid_rank,REVIEW_REFRESH
7,8,content_f6116743b00afc2d,81123.0,10.0,0.012,9.857661,visible_mid_rank,REVIEW_REFRESH
8,9,content_f43118e089ecc69a,80299.0,129.0,0.161,4.853248,visible_mid_rank,REVIEW_REFRESH
9,10,content_95ff62babbfac9c7,78330.0,155.0,0.198,4.581667,visible_mid_rank,REVIEW_REFRESH


### Top-10 manual review

1. **Rank 1 — REVIEW_REFRESH** — Very high search visibility (179,025 impressions) and an average position of 15.6 place this page clearly inside the rule's opportunity range. It would be a wrong pick if manual review shows that the page is already current and its lower position is driven by factors that a content refresh cannot reasonably improve.

2. **Rank 2 — REVIEW_REFRESH** — High visibility (127,816 impressions) and a strong average position of 4.3 make this a high-impact candidate near the top of the eligible range. It would be a wrong pick if the page is already performing as intended and there is little realistic room for improvement from a refresh.

3. **Rank 3 — REVIEW_REFRESH** — The page has 125,082 impressions and an average position of 4.4, so it has meaningful visibility and remains eligible under the rule. It would be a wrong pick if the observed visibility already reflects an optimized page with no actionable content issue.

4. **Rank 4 — REVIEW_REFRESH** — With 110,663 impressions and an average position of 4.5, the page has substantial search exposure and sits just outside the very top positions. It would be a wrong pick if a manual review finds no refreshable content problem.

5. **Rank 5 — REVIEW_REFRESH** — The page has 100,270 impressions but a very low CTR of 0.057%, making it worth reviewing as a potentially weak conversion of visibility into clicks. It would be a wrong pick if the low CTR is normal for the queries or search-result context in which the page appears.

6. **Rank 6 — REVIEW_REFRESH** — The page has 96,942 impressions and an average position of 18.3, so it combines high visibility with a relatively weak search position. It would be a wrong pick if the ranking gap is caused mainly by factors outside the content itself.

7. **Rank 7 — REVIEW_REFRESH** — The page has 89,534 impressions and an average position of 17.8, making it a visible but lower-ranked candidate. It would be a wrong pick if the page is already current and its position is not realistically addressable through a refresh.

8. **Rank 8 — REVIEW_REFRESH** — The page has 81,123 impressions but only 10 clicks, producing an extremely low CTR of 0.012%. This makes it especially worth inspecting. It would be a wrong pick if the low CTR is expected for the page's search context rather than a content or search-snippet problem.

9. **Rank 9 — REVIEW_REFRESH** — The page has 80,299 impressions and an average position of 4.9, so it has strong visibility and remains inside the rule's opportunity range. It would be a wrong pick if the page is already close to its realistic search ceiling and no useful refresh opportunity exists.

10. **Rank 10 — REVIEW_REFRESH** — The page has 78,330 impressions and an average position of 4.6, giving it substantial exposure and a strong but not top-most average position. It would be a wrong pick if manual review finds that the content is already current and well aligned with its search intent.

## 4. Weak picks + leakage check

### Weak picks

Ranks 5, 6, and 8 deserve extra skepticism.

- **Rank 5** has 100,270 impressions but a CTR of only 0.057%. The page has strong visibility, but the low click rate suggests that high exposure alone may not mean the page is a strong refresh opportunity.
- **Rank 6** has 96,942 impressions, a CTR of 0.039%, and an average position of 18.3. It qualifies under the rule, but its weaker position may reflect a broader ranking problem that a simple content refresh may not solve.
- **Rank 8** has 81,123 impressions but only 10 clicks, for a CTR of 0.012%. This is the clearest weak pick in the top ten because the rule rewards visibility even when that visibility produces almost no clicks.

These cases show a limitation of the baseline: it ranks opportunity mainly by visibility after applying the position filter, so it can prioritize pages whose impressions do not translate into meaningful search traffic.

In [10]:
# Verify that the ranked baseline queue contains only pre-decision inputs.
# Future-window and label-derived columns must not appear in the final queue.

forbidden_columns = {
    "future_impressions_10d",
    "future_change_pct",
    "future_decline_target",
}

leaked_columns = sorted(
    forbidden_columns.intersection(ranked_queue.columns)
)

print("Leaked future/label columns:", leaked_columns)
print("Leakage check passed:", len(leaked_columns) == 0)
print("Final queue columns:", list(ranked_queue.columns))


Leaked future/label columns: []
Leakage check passed: True
Final queue columns: ['rank', 'client_hash_id', 'content_hash_id', 'impressions_21d', 'clicks_21d', 'avg_position_21d', 'baseline_score', 'reason_code', 'action_label']


### Leakage result

The ranked baseline queue contains no future-window or label-derived columns.

Only pre-decision search signals are used to select and rank pages. The future outcome fields created for the signal checks remain outside the baseline queue and are used only for analysis.

## 5. Self-check

- [x] I checked two pre-decision signals with visible bucket tables and `n`.
- [x] At least one checked signal is linked to a real FlyRank flag logic.
- [x] Each signal check has a one-word verdict.
- [x] I wrote one transparent baseline rule with a score, one reason code, and one action label.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] I manually reviewed the top 10 and stated what could make each pick wrong.
- [x] I identified weak picks without changing the baseline after seeing the results.
- [x] No future-window or label-derived inputs are used in the ranked queue.
- [x] No client names, URLs, or private queries appear in the notebook.
- [x] My claims use careful decision-support language.